# Hybrid Model Zoo + Validation-Selected Ensemble for MemTrOC Rejection Prediction

This notebook keeps the existing **edge-aware GINE + tabular model** and adds complementary molecular/tabular regressors:
- ExtraTrees and Random Forest
- HistGradientBoosting
- CatBoost
- Validation-selected single model and pairwise ensemble

Tree models receive the 19 tabular variables plus Morgan fingerprints and RDKit 2D descriptors. The ensemble is selected **using validation RMSE only**; test results are reported afterward. The notebook keeps the original random-row split for comparability with the prior run, so duplicate SMILES can cross splits. Use a separate grouped split if the goal is unseen-molecule generalization.

No model is guaranteed to beat the existing result. This experiment is designed to test whether complementary model families improve it fairly.

**Before running**
1. In Colab, choose Runtime → Change runtime type → T4 GPU (recommended for GINE).
2. Run cells from top to bottom.
3. Upload the original `MolGBN-OPR-master.zip` when prompted.
4. Compare all candidates on the same validation and test partitions.


In [ ]:
# 1. Install dependencies
%pip -q install torch-geometric ogb rdkit scikit-learn pandas joblib catboost


In [ ]:
# 2. Upload your existing project ZIP
from google.colab import files
uploaded = files.upload()  # Select MolGBN-OPR-master.zip


In [ ]:
# 3. Extract the project and locate the dataset
import os, zipfile, glob
from pathlib import Path

zip_names = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
if not zip_names:
    raise FileNotFoundError("Please upload MolGBN-OPR-master.zip.")

project_zip = zip_names[0]
extract_dir = Path("/content/molgbn_project")
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(project_zip, "r") as zf:
    zf.extractall(extract_dir)

matches = list(extract_dir.rglob("MemTrOC-Dataset.csv"))
if not matches:
    raise FileNotFoundError(
        "Could not find MemTrOC-Dataset.csv inside the uploaded ZIP. "
        "Check that you uploaded the original MolGBN-OPR project ZIP."
    )
DATA_PATH = matches[0]
PROJECT_ROOT = DATA_PATH.parents[2]  # .../MolGBN-OPR-master/data/processed/file.csv
OUTPUT_DIR = Path("/content/edge_aware_gnn_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATA_PATH)
print("Project root:", PROJECT_ROOT)
print("Outputs:", OUTPUT_DIR)


In [ ]:
# 4. Imports and reproducibility
import copy, json, random, math
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Batch, Data
from torch_geometric.nn import GINEConv, global_mean_pool
from ogb.utils import smiles2graph
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import joblib

SEED = 41
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cpu":
    print("Tip: Runtime → Change runtime type → T4 GPU can speed up training.")


In [ ]:
# 5. Edge-aware Tabular + GNN architecture
# OGB categorical atom/bond feature cardinalities, matching the project's
# src/utils/smiles2graph.py graph builder.
ATOM_FEATURE_CARDINALITIES = (119, 4, 12, 12, 10, 6, 6, 2, 2)
BOND_FEATURE_CARDINALITIES = (5, 6, 2)

class CategoricalFeatureEncoder(nn.Module):
    """Embed each categorical feature column and sum embeddings per atom/bond."""
    def __init__(self, cardinalities, hidden_dim):
        super().__init__()
        self.cardinalities = tuple(cardinalities)
        self.embeddings = nn.ModuleList(
            nn.Embedding(size, hidden_dim) for size in self.cardinalities
        )

    def forward(self, features):
        features = features.long()
        if features.ndim != 2 or features.shape[1] != len(self.embeddings):
            raise ValueError(
                f"Expected {len(self.embeddings)} categorical columns; "
                f"received shape {tuple(features.shape)}"
            )
        out = 0
        for col, embedding in enumerate(self.embeddings):
            ids = features[:, col].clamp(min=0, max=self.cardinalities[col] - 1)
            out = out + embedding(ids)
        return out

class EdgeAwareTabularGNN(nn.Module):
    def __init__(self, table_input_dim=19, hidden_dim=128, dropout=0.20):
        super().__init__()
        self.dropout = dropout
        self.table_net = nn.Sequential(
            nn.Linear(table_input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.atom_encoder = CategoricalFeatureEncoder(ATOM_FEATURE_CARDINALITIES, hidden_dim)
        self.bond_encoder = CategoricalFeatureEncoder(BOND_FEATURE_CARDINALITIES, hidden_dim)

        mlp1 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        mlp2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.gnn1 = GINEConv(mlp1, edge_dim=hidden_dim)
        self.gnn2 = GINEConv(mlp2, edge_dim=hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        # Concatenation preserves both branches' information.
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, table_x, graph):
        table_emb = self.table_net(table_x)
        atom_emb = self.atom_encoder(graph.x)
        bond_emb = self.bond_encoder(graph.edge_attr)

        x = self.gnn1(atom_emb, graph.edge_index, bond_emb)
        x = self.norm1(F.relu(x) + atom_emb)
        x = F.dropout(x, p=self.dropout, training=self.training)

        residual = x
        x = self.gnn2(x, graph.edge_index, bond_emb)
        x = self.norm2(F.relu(x) + residual)

        graph_emb = global_mean_pool(x, graph.batch)
        fused = torch.cat([table_emb, graph_emb], dim=1)
        return self.head(fused)


In [ ]:
# 6. Load and prepare the existing dataset
df_all = pd.read_csv(DATA_PATH)
print("Original dataset shape:", df_all.shape)
print("Columns:")
print(list(enumerate(df_all.columns)))

# Match the original project training code:
# SMILES = column 3, tabular features = columns 4:23 (19 features),
# target = column 23 (TrOC Rejection %).
smiles_col = df_all.columns[3]
feature_cols = list(df_all.columns[4:23])
target_col = df_all.columns[23]

df = df_all[[smiles_col] + feature_cols + [target_col]].copy()
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
df[smiles_col] = df[smiles_col].astype("string").str.strip()
df = df.dropna(subset=[smiles_col, target_col])
df = df[df[smiles_col].notna() & df[smiles_col].ne("")]
df = df.reset_index(drop=True)

smiles = df[smiles_col].astype(str).to_numpy()
X_raw = df[feature_cols].to_numpy(dtype=np.float64)
y = df[target_col].to_numpy(dtype=np.float32)

print(f"Usable rows: {len(df)}")
print(f"SMILES column: {smiles_col}")
print(f"Number of tabular features: {len(feature_cols)}")
print(f"Target: {target_col}")
display(df[[smiles_col, target_col]].head())


In [ ]:
# 7. Split data first, then fit preprocessing on training rows only
# 70% train / 20% validation / 10% test.
all_idx = np.arange(len(df))
train_val_idx, test_idx = train_test_split(
    all_idx, test_size=0.10, random_state=SEED
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=(0.20 / 0.90), random_state=SEED
)

imputer = SimpleImputer(strategy="median")
scaler = MinMaxScaler()

X_train = scaler.fit_transform(imputer.fit_transform(X_raw[train_idx])).astype(np.float32)
X_val = scaler.transform(imputer.transform(X_raw[val_idx])).astype(np.float32)
X_test = scaler.transform(imputer.transform(X_raw[test_idx])).astype(np.float32)

print(f"Train: {len(train_idx)} | Validation: {len(val_idx)} | Test: {len(test_idx)}")


In [ ]:
# 8. Convert SMILES to OGB molecular graphs and cache them
# This can take a few minutes. Graphs are created once, not every epoch.
graphs = []
invalid_rows = []

for i, smi in enumerate(smiles):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        invalid_rows.append(i)
        graphs.append(None)
        continue
    graph_dict = smiles2graph(smi)
    x = torch.as_tensor(graph_dict["node_feat"], dtype=torch.long)
    edge_index = torch.as_tensor(graph_dict["edge_index"], dtype=torch.long)
    edge_attr = torch.as_tensor(graph_dict["edge_feat"], dtype=torch.long)

    # OGB returns edge features for directed edges. For a molecule without bonds,
    # ensure the expected 2D empty shape (0, 3).
    if edge_attr.numel() == 0:
        edge_attr = edge_attr.reshape(0, len(BOND_FEATURE_CARDINALITIES))
    graphs.append(Data(x=x, edge_index=edge_index, edge_attr=edge_attr))

if invalid_rows:
    raise ValueError(
        f"Found {len(invalid_rows)} invalid SMILES rows after filtering. "
        f"First row indexes: {invalid_rows[:10]}. Please inspect the dataset."
    )

print("Cached molecular graphs:", len(graphs))
print("Atom feature shape:", tuple(graphs[0].x.shape))
print("Bond feature shape:", tuple(graphs[0].edge_attr.shape))


In [ ]:
# 9. Dataset, batching, and metric helpers
class TableGraphDataset(Dataset):
    def __init__(self, table_x, graph_list, targets):
        self.table_x = torch.as_tensor(table_x, dtype=torch.float32)
        self.graphs = graph_list
        self.targets = torch.as_tensor(targets, dtype=torch.float32).reshape(-1)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        return self.table_x[i], self.graphs[i], self.targets[i]

def collate_table_graph(items):
    tables, graphs_batch, targets = zip(*items)
    return (
        torch.stack(tables, dim=0),
        Batch.from_data_list(list(graphs_batch)),
        torch.stack(targets).reshape(-1, 1)
    )

train_ds = TableGraphDataset(X_train, [graphs[i] for i in train_idx], y[train_idx])
val_ds = TableGraphDataset(X_val, [graphs[i] for i in val_idx], y[val_idx])
test_ds = TableGraphDataset(X_test, [graphs[i] for i in test_idx], y[test_idx])

BATCH_SIZE = 64
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_table_graph,
    num_workers=0, generator=torch.Generator().manual_seed(SEED)
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_table_graph,
    num_workers=0
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_table_graph,
    num_workers=0
)

def calculate_metrics(y_true, y_pred):
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred))
    }

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for table_x, graph, target in loader:
        table_x = table_x.to(DEVICE)
        graph = graph.to(DEVICE)
        pred = model(table_x, graph)
        y_true.extend(target.numpy().reshape(-1).tolist())
        y_pred.extend(pred.detach().cpu().numpy().reshape(-1).tolist())
    return calculate_metrics(np.asarray(y_true), np.asarray(y_pred)), np.asarray(y_true), np.asarray(y_pred)


In [ ]:
# 10. Train with validation-based early stopping
# You can reduce MAX_EPOCHS for a quick smoke test; 300 is the intended first run.
MAX_EPOCHS = 300
PATIENCE = 40
HIDDEN_DIM = 128
DROPOUT = 0.20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

model = EdgeAwareTabularGNN(
    table_input_dim=len(feature_cols),
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10
)
loss_fn = nn.MSELoss()

best_val_rmse = float("inf")
best_epoch = 0
epochs_no_improve = 0
best_state = None
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    loss_sum, n_seen = 0.0, 0

    for table_x, graph, target in train_loader:
        table_x = table_x.to(DEVICE)
        graph = graph.to(DEVICE)
        target = target.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        prediction = model(table_x, graph)
        loss = loss_fn(prediction, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        loss_sum += loss.item() * len(target)
        n_seen += len(target)

    train_rmse = math.sqrt(loss_sum / max(n_seen, 1))
    val_metrics, _, _ = evaluate(model, val_loader)
    scheduler.step(val_metrics["RMSE"])

    history.append({
        "epoch": epoch,
        "train_RMSE": train_rmse,
        "val_R2": val_metrics["R2"],
        "val_RMSE": val_metrics["RMSE"],
        "val_MAE": val_metrics["MAE"],
        "learning_rate": optimizer.param_groups[0]["lr"]
    })

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | train RMSE={train_rmse:.4f} | "
            f"val R2={val_metrics['R2']:.4f} | val RMSE={val_metrics['RMSE']:.4f}"
        )

    if val_metrics["RMSE"] < best_val_rmse:
        best_val_rmse = val_metrics["RMSE"]
        best_epoch = epoch
        epochs_no_improve = 0
        best_state = copy.deepcopy(model.state_dict())
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch}; best epoch={best_epoch}.")
        break

if best_state is None:
    raise RuntimeError("Training did not produce a valid model state.")

model.load_state_dict(best_state)
print("Best epoch:", best_epoch)
print("Best validation RMSE:", round(best_val_rmse, 4))


In [ ]:
# 11. Final test evaluation (run only after model selection)
val_metrics, val_y_true, val_y_pred = evaluate(model, val_loader)
test_metrics, test_y_true, test_y_pred = evaluate(model, test_loader)

results = {
    "model": "EdgeAwareTabularGNN (GINE + bond features + concatenation)",
    "seed": SEED,
    "split": {
        "method": "random row split",
        "train_rows": int(len(train_idx)),
        "validation_rows": int(len(val_idx)),
        "test_rows": int(len(test_idx))
    },
    "best_epoch": int(best_epoch),
    "validation": val_metrics,
    "test": test_metrics,
    "hyperparameters": {
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE
    },
    "warning": (
        "A random row split can put the same SMILES in both train and test. "
        "This evaluates rows for familiar molecules as well as new conditions; "
        "use a separate SMILES-grouped split to measure unseen-molecule generalization."
    )
}

print("\nVALIDATION:", json.dumps(val_metrics, indent=2))
print("\nHELD-OUT TEST:", json.dumps(test_metrics, indent=2))


In [ ]:
# 12. Train complementary models and select a validation-only ensemble
# This cell does not use test performance to choose the model or blend weights.
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from catboost import CatBoostRegressor
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit import DataStructs
from itertools import combinations

print("Building molecular descriptors and Morgan fingerprints...")
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

fingerprints = []
descriptor_rows = []
for smi in smiles:
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        raise ValueError(f"Invalid SMILES encountered during feature generation: {smi}")
    fp = morgan_generator.GetFingerprint(mol)
    fp_array = np.zeros((2048,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, fp_array)
    fingerprints.append(fp_array)
    values = []
    for _, calc in Descriptors.descList:
        try:
            values.append(float(calc(mol)))
        except Exception:
            values.append(np.nan)
    descriptor_rows.append(values)

X_fp = np.asarray(fingerprints, dtype=np.float32)
X_desc = np.asarray(descriptor_rows, dtype=np.float64)
X_desc[~np.isfinite(X_desc)] = np.nan
# Drop descriptor columns that are entirely missing; remaining missing values
# are imputed from the training partition only.
keep_desc_cols = ~np.all(np.isnan(X_desc), axis=0)
X_desc = X_desc[:, keep_desc_cols]
if X_desc.shape[1] == 0:
    raise ValueError('No usable RDKit descriptors were generated.')

table_imputer = SimpleImputer(strategy="median")
X_table_imp = np.empty_like(X_raw, dtype=np.float32)
X_table_imp[train_idx] = table_imputer.fit_transform(X_raw[train_idx]).astype(np.float32)
X_table_imp[val_idx] = table_imputer.transform(X_raw[val_idx]).astype(np.float32)
X_table_imp[test_idx] = table_imputer.transform(X_raw[test_idx]).astype(np.float32)

desc_imputer = SimpleImputer(strategy="median")
X_desc_imp = np.empty_like(X_desc, dtype=np.float32)
X_desc_imp[train_idx] = desc_imputer.fit_transform(X_desc[train_idx]).astype(np.float32)
X_desc_imp[val_idx] = desc_imputer.transform(X_desc[val_idx]).astype(np.float32)
X_desc_imp[test_idx] = desc_imputer.transform(X_desc[test_idx]).astype(np.float32)

X_mol = np.concatenate([X_table_imp, X_fp, X_desc_imp], axis=1)
Xtr, Xva, Xte = X_mol[train_idx], X_mol[val_idx], X_mol[test_idx]
ytr, yva, yte = y[train_idx], y[val_idx], y[test_idx]
print("Tree-model feature matrix:", X_mol.shape)

candidate_models = {
    "ExtraTrees_Morgan_RDKit_Tabular": ExtraTreesRegressor(
        n_estimators=700, min_samples_leaf=1, max_features=0.85,
        n_jobs=-1, random_state=SEED
    ),
    "RandomForest_Morgan_RDKit_Tabular": RandomForestRegressor(
        n_estimators=600, min_samples_leaf=1, max_features=0.85,
        n_jobs=-1, random_state=SEED
    ),
    "HistGradientBoosting_Morgan_RDKit_Tabular": HistGradientBoostingRegressor(
        max_iter=350, learning_rate=0.04, max_leaf_nodes=15,
        l2_regularization=1.0, early_stopping=True,
        validation_fraction=0.15, random_state=SEED
    ),
    "CatBoost_Morgan_RDKit_Tabular": CatBoostRegressor(
        iterations=1200, depth=6, learning_rate=0.03,
        loss_function="RMSE", eval_metric="RMSE",
        l2_leaf_reg=5.0, random_seed=SEED, verbose=False,
        allow_writing_files=False
    ),
}

val_predictions = {
    "EdgeAware_GINE": np.asarray(val_y_pred).reshape(-1)
}
test_predictions = {
    "EdgeAware_GINE": np.asarray(test_y_pred).reshape(-1)
}
model_objects = {}

for name, estimator in candidate_models.items():
    print(f"\nFitting {name} ...")
    if name.startswith("CatBoost"):
        estimator.fit(
            Xtr, ytr,
            eval_set=(Xva, yva),
            early_stopping_rounds=100,
            verbose=False
        )
    else:
        estimator.fit(Xtr, ytr)
    model_objects[name] = estimator
    val_predictions[name] = np.asarray(estimator.predict(Xva)).reshape(-1)
    test_predictions[name] = np.asarray(estimator.predict(Xte)).reshape(-1)

# Include all single candidates, ranked using validation RMSE.
val_rows = []
for name, pred in val_predictions.items():
    metrics = calculate_metrics(yva, pred)
    val_rows.append({"model": name, **metrics})
val_comparison = pd.DataFrame(val_rows).sort_values("RMSE").reset_index(drop=True)

print("\nValidation comparison (used for selection):")
display(val_comparison)

# Search single models and pairwise blends on validation only.
# Grid weights are intentionally coarse to reduce excessive tuning to validation.
best_choice = {
    "name": None,
    "val_rmse": float("inf"),
    "members": None,
    "weights": None,
    "val_pred": None,
}
for name, pred in val_predictions.items():
    rmse = float(np.sqrt(mean_squared_error(yva, pred)))
    if rmse < best_choice["val_rmse"]:
        best_choice = {
            "name": name, "val_rmse": rmse,
            "members": [name], "weights": [1.0],
            "val_pred": pred.copy()
        }

names = list(val_predictions.keys())
for name_a, name_b in combinations(names, 2):
    pred_a, pred_b = val_predictions[name_a], val_predictions[name_b]
    for weight_a in np.linspace(0.0, 1.0, 21):
        blend_pred = weight_a * pred_a + (1.0 - weight_a) * pred_b
        rmse = float(np.sqrt(mean_squared_error(yva, blend_pred)))
        if rmse < best_choice["val_rmse"]:
            best_choice = {
                "name": f"Blend({name_a}, {name_b})",
                "val_rmse": rmse,
                "members": [name_a, name_b],
                "weights": [float(weight_a), float(1.0 - weight_a)],
                "val_pred": blend_pred.copy()
            }

chosen_test_pred = sum(
    w * test_predictions[name]
    for name, w in zip(best_choice["members"], best_choice["weights"])
)
chosen_val_pred = best_choice["val_pred"]

# Report test metrics after the validation-based choice has been finalized.
test_rows = []
for name, pred in test_predictions.items():
    test_rows.append({
        "model": name,
        **calculate_metrics(yte, pred),
        "selected_by_validation": name == best_choice["name"]
    })
test_rows.append({
    "model": best_choice["name"],
    **calculate_metrics(yte, chosen_test_pred),
    "selected_by_validation": True
})
test_comparison = pd.DataFrame(test_rows).sort_values("RMSE").reset_index(drop=True)

ensemble_results = {
    "selection_rule": "Lowest validation RMSE among single candidates and pairwise blends",
    "selected_model_or_blend": best_choice["name"],
    "selected_members": best_choice["members"],
    "selected_weights": best_choice["weights"],
    "validation_metrics": calculate_metrics(yva, chosen_val_pred),
    "test_metrics": calculate_metrics(yte, chosen_test_pred),
    "candidate_validation_metrics": val_comparison.to_dict(orient="records"),
    "candidate_test_metrics": test_comparison.to_dict(orient="records"),
    "split": {
        "method": "random row split",
        "train_rows": int(len(train_idx)),
        "validation_rows": int(len(val_idx)),
        "test_rows": int(len(test_idx))
    },
    "warning": (
        "Test metrics are reported after model selection using validation RMSE. "
        "Do not adjust weights or hyperparameters based on test performance. "
        "Random row split may allow duplicate canonical SMILES across subsets."
    )
}

print("\nSelected using validation only:")
print(best_choice["name"])
print("Members:", best_choice["members"])
print("Weights:", best_choice["weights"])
print("Validation metrics:", ensemble_results["validation_metrics"])
print("\nTest comparison:")
display(test_comparison)

# Persist separate challenger outputs; original GINE files are preserved.
with open(OUTPUT_DIR / "hybrid_ensemble_results.json", "w") as f:
    json.dump(ensemble_results, f, indent=2)

pd.DataFrame({
    "y_true": yte,
    "gine_prediction": test_predictions["EdgeAware_GINE"],
    "selected_prediction": chosen_test_pred,
    "selected_residual": yte - chosen_test_pred
}).to_csv(OUTPUT_DIR / "hybrid_ensemble_test_predictions.csv", index=False)

pd.DataFrame(val_rows).to_csv(
    OUTPUT_DIR / "hybrid_candidate_validation_metrics.csv", index=False
)
test_comparison.to_csv(
    OUTPUT_DIR / "hybrid_candidate_test_metrics.csv", index=False
)
joblib.dump(
    {
        "models": model_objects,
        "selected_members": best_choice["members"],
        "selected_weights": best_choice["weights"],
        "table_imputer": table_imputer,
        "descriptor_imputer": desc_imputer,
        "feature_columns": feature_cols,
        "target_column": target_col,
        "smiles_column": smiles_col,
        "morgan_radius": 2,
        "morgan_fp_size": 2048,
    },
    OUTPUT_DIR / "hybrid_ensemble_models.joblib"
)
print("Saved ensemble comparison and predictions to:", OUTPUT_DIR)


In [ ]:
# 13. Save model, preprocessing, metrics, history, and test predictions
checkpoint_path = OUTPUT_DIR / "best_edge_aware_tabular_gnn.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "table_input_dim": len(feature_cols),
    "hidden_dim": HIDDEN_DIM,
    "dropout": DROPOUT,
    "feature_columns": feature_cols,
    "target_column": target_col,
    "smiles_column": smiles_col,
    "seed": SEED,
    "best_epoch": best_epoch,
    "best_val_rmse": best_val_rmse
}, checkpoint_path)

with open(OUTPUT_DIR / "metrics.json", "w") as f:
    json.dump(results, f, indent=2)

pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
pd.DataFrame({
    "y_true": test_y_true,
    "y_pred": test_y_pred,
    "residual": test_y_true - test_y_pred
}).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

joblib.dump({
    "imputer": imputer,
    "scaler": scaler,
    "feature_columns": feature_cols
}, OUTPUT_DIR / "preprocessing.joblib")

print("Saved outputs to:", OUTPUT_DIR)
print("Files:", sorted(p.name for p in OUTPUT_DIR.iterdir()))


In [ ]:
# 14. Download the outputs as a ZIP
import shutil
archive_path = shutil.make_archive("/content/edge_aware_gnn_outputs", "zip", OUTPUT_DIR)
print("Created:", archive_path)

from google.colab import files
files.download(archive_path)


## How to interpret the result

- **Higher R²** is better; **lower RMSE and MAE** are better.
- The new tree models use tabular inputs plus Morgan fingerprints and RDKit descriptors, complementing the GINE model.
- The selected single model or pairwise blend is chosen using **validation RMSE only**. The held-out test set is not used to choose blend weights.
- Use `hybrid_candidate_validation_metrics.csv` to see selection performance and `hybrid_candidate_test_metrics.csv` for final held-out comparisons.
- This notebook retains a random-row split to compare with the previous run. Repeated SMILES can occur across splits, so results do not prove generalization to unseen molecules.
- For a defensible final report, run a separate canonical-SMILES-grouped experiment and keep its test set untouched during tuning.
- No model is guaranteed to beat the current best. Keep the existing GINE checkpoint/results if the challenger does not improve the same-split metrics.
